# GCOL-OBS-0.3.2.1.1-PATCH.10H.1 — Continuum-only geometry & FITS metadata preflight

**Project** ALMA 2015.1.00147.S · **Source** NGC 1333 IRAS 4A · **Members** X22 and X24

**Mandatory upstream status: `PATCH10H0_PASS = False`; `NEXT_PATCH_AUTHORIZED = False`.** This notebook is a **conditional preflight**. It does not override the upstream fail, and must not claim the full PATCH.10H.1 gate passed. It can register real **X24-only continuum** positions and identify blockers.

**Blindness / scientific boundaries**

- Read **one X24 combined continuum FITS** and the previously archived PATCH10H0 metadata CSV/JSON only. Never read spectral cube pixel arrays or inspect line signal amplitudes.
- Keep pre-registered windows `core = [6.40, 7.00] km/s` and `foreground_test = [7.70, 8.30] km/s` **exactly** unchanged.
- Retain original `EXPECTED_SPW` labels unmodified; report any FITS `RESTFRQ` mismatch without silently updating the scientific contract.
- Never claim foreground/background layer, line detection, a GVH prediction, or CHH-derived physics: `GVH_USED = False`.
- Output a sky-coordinate aperture recipe **only**; convert it to each individual cube's pixel geometry later, using that cube's own full spatial WCS metadata, after provenance issues are resolved.
- No intrinsic derivatives belong to the CHH composition framework; this is an **independent observational preparation branch**, not dynamic GVH.

**Source references:** original archived `PATCH10H0_*.json/csv` from the user's Drive; J. Choi et al., *Radio Imaging of the NGC 1333 IRAS 4A Region*, PASJ 63 (2011), Table 1 (external comparison only). Positions from the external publication are **not used to select peaks**.

In [1]:
from pathlib import Path
import os, json, hashlib, math
import numpy as np
import pandas as pd
from scipy.ndimage import maximum_filter
from scipy.optimize import linear_sum_assignment
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

PATCH = 'GCOL-OBS-0.3.2.1.1-PATCH.10H.1'
PREFIX = 'GCOL_OBS_0_3_2_1_1_PATCH10H1_Continuum_Geometry_and_Spectral_Metadata_Preflight_FAST'
LOCAL = Path('/mnt/data') if Path('/mnt/data/PATCH10H0_status.json').is_file() else Path.cwd()
# Standard Colab fallback. The original Drive folder is read-only here; no source mutation.
DRIVE = Path('/content/drive/MyDrive/GCOL_OBS_0_3_2_1_1_PATCH10H0')
if not (LOCAL/'PATCH10H0_status.json').exists() and DRIVE.exists():
    LOCAL = DRIVE
INPUT_CANDIDATES = sorted(Path('/mnt/data').glob('*X24*cont.I.pbcor.fits.bin')) + sorted((DRIVE/'ALMA_2015.1.00147.S'/'X24').glob('*cont.I.pbcor.fits')) if DRIVE.exists() else sorted(Path('/mnt/data').glob('*X24*cont.I.pbcor.fits.bin'))
INPUT_CANDIDATES += sorted(Path.cwd().glob('*X24*cont.I.pbcor.fits*'))
FITS_FILE = next((p for p in INPUT_CANDIDATES if p.is_file()), None)
if FITS_FILE is None:
    raise FileNotFoundError('X24 combined continuum missing. Mount your Drive or unpack the supplied reproducibility ZIP into /mnt/data before running.')
OUT = Path('/mnt/data') if Path('/mnt/data').exists() else Path.cwd()
print('Continuum input:', FITS_FILE.name)
print('Metadata root:', LOCAL)

Continuum input: member.uid___A001_X2f6_X24.NGC1333_IRAS4A_sci.spw25_27_29_31_33_35_37.cont.I.pbcor.fits.bin
Metadata root: /mnt/data


In [2]:
CONTRACT = json.loads((LOCAL/'PATCH10H0_preregistration_contract.json').read_text())
H0 = json.loads((LOCAL/'PATCH10H0_status.json').read_text())
H0_HEADER = pd.read_csv(LOCAL/'PATCH10H0_fits_header_audit.csv')
H0_REQUIRED = pd.read_csv(LOCAL/'PATCH10H0_required_set_audit.csv')
H0_HASH = pd.read_csv(LOCAL/'PATCH10H0_required_files_sha256.csv')
ORIGINAL_CONTRACT_SHA256 = hashlib.sha256((LOCAL/'PATCH10H0_preregistration_contract.json').read_bytes()).hexdigest()
FROZEN = CONTRACT['velocity_windows_kms']
assert FROZEN == {'core':[6.4,7.0],'foreground_test':[7.7,8.3]}
assert CONTRACT['primary_spws'] == [27,29,31,33,35,37]
assert CONTRACT['GVH_used'] is False
assert H0['PATCH10H0_PASS'] is False and H0['NEXT_PATCH_AUTHORIZED'] is False
assert H0['X22_REQUIRED_SET_PASS'] is False and H0['README_PRESENT'] is False
assert H0['X24_REQUIRED_SET_PASS'] is True
print('UPSTREAM GATE:', H0['PATCH10H0_DECISION'])
print('X22 continuum archived count:',int(H0_REQUIRED.set_index('member').loc['X22','continuum_pbcor_count']))
print('X24 continuum archived count:',int(H0_REQUIRED.set_index('member').loc['X24','continuum_pbcor_count']))

UPSTREAM GATE: ABSTAIN_ARCHIVE_OR_PROVENANCE_NOT_READY
X22 continuum archived count: 0
X24 continuum archived count: 1


## Level 1 — FITS integrity and exact SIN celestial WCS

Only the combined X24 continuum has pixels read. We use a **narrow FITS decoder** for this exact original 4D float32 image with singleton FREQ/STOKES axes, and exact SIN projection with identity PC and zero slant PV. Unsupported layouts are rejected, never guessed. For generalized WCS in future cube operations, use Astropy/WCS after obtaining complete cube headers. This code does **not** claim to be a universal WCS implementation.

In [3]:
def read_fits_continuum(path):
    with open(path,'rb') as f:
        cards=[]
        while True:
            block=f.read(2880)
            if len(block)!=2880: raise ValueError('truncated FITS header')
            stop=False
            for j in range(0,2880,80):
                card=block[j:j+80].decode('ascii')
                cards.append(card)
                if card.startswith('END '): stop=True;break
            if stop: break
        offset=f.tell()
    H={}
    for card in cards:
        if card[8:10]!='= ':continue
        key=card[:8].strip()
        val=card[10:80].lstrip()
        if val.startswith("'"):
            v=val.split("'",2)[1].strip()
        else:
            v=val.split('/',1)[0].strip()
            if v in ('T','F'):v=(v=='T')
            else:
                try:v=float(v.replace('D','E'))
                except (ValueError,TypeError):pass
        H[key]=v
    assert int(H['BITPIX'])==-32 and int(H['NAXIS'])==4 and int(H['NAXIS3'])==int(H['NAXIS4'])==1
    nx,ny=int(H['NAXIS1']),int(H['NAXIS2'])
    assert H['CTYPE1']=='RA---SIN' and H['CTYPE2']=='DEC--SIN'
    assert H['CTYPE3']=='FREQ' and H['CTYPE4']=='STOKES'
    assert H['CUNIT1']=='deg' and H['CUNIT2']=='deg'
    for i in range(1,5):
        for j in range(1,5):
            if f'PC{i:02d}_{j:02d}' in H:
                assert np.isclose(H[f'PC{i:02d}_{j:02d}'],int(i==j))
    assert abs(float(H.get('PV2_1',0)))<1e-12 and abs(float(H.get('PV2_2',0)))<1e-12
    arr=np.memmap(path,dtype='>f4',mode='r',offset=offset,shape=(ny,nx))
    assert path.stat().st_size>=offset+nx*ny*4
    return H,arr,offset

H,arr,data_offset = read_fits_continuum(FITS_FILE)
image=np.asarray(arr,dtype=np.float64)
finite=np.isfinite(image)
checksum=hashlib.sha256(FITS_FILE.read_bytes()).hexdigest()
hrows=H0_HASH[(H0_HASH['member']=='X24') & (H0_HASH['name'].str.contains('cont.I.pbcor.fits',regex=False))]
assert len(hrows)==1 and checksum==str(hrows.iloc[0]['sha256'])
assert H['OBJECT']=='NGC1333_IRAS4A' and H['BUNIT']=='Jy/beam' and H['SPECSYS']=='LSRK'
print('X24 SHA-256 archived MATCH:',checksum)
print('Size:',image.shape,'valid pixels:',int(finite.sum()),'beam arcsec:',float(H['BMAJ'])*3600,float(H['BMIN'])*3600)

X24 SHA-256 archived MATCH: e858a7fd8fafde20799082c60c37c1c17e0d0f3b7fbe402a8dcf75a781c63b1c
Size: (320, 320) valid pixels: 58897 beam arcsec: 0.65608811378496 0.46625125408164003


In [4]:
def pixel_to_sky_sin(x,y,H):
    # FITS 1-indexed CRPIX vs 0-indexed numpy pixels; no slant correction, PC identity
    l=np.deg2rad((np.asarray(x)+1-float(H['CRPIX1']))*float(H['CDELT1']))
    m=np.deg2rad((np.asarray(y)+1-float(H['CRPIX2']))*float(H['CDELT2']))
    n=np.sqrt(1-l*l-m*m)
    ra0=np.deg2rad(float(H['CRVAL1']));dec0=np.deg2rad(float(H['CRVAL2']))
    ra=(ra0+np.arctan2(l,n*np.cos(dec0)-m*np.sin(dec0)))%(2*np.pi)
    dec=np.arcsin(m*np.cos(dec0)+n*np.sin(dec0))
    return np.rad2deg(ra),np.rad2deg(dec)

def sky_to_pixel_sin(ra_deg,dec_deg,H):
    ra,dec=np.deg2rad(ra_deg),np.deg2rad(dec_deg)
    ra0,dec0=np.deg2rad(float(H['CRVAL1'])),np.deg2rad(float(H['CRVAL2']))
    dra=ra-ra0
    l=np.cos(dec)*np.sin(dra)
    m=np.sin(dec)*np.cos(dec0)-np.cos(dec)*np.sin(dec0)*np.cos(dra)
    x=np.rad2deg(l)/float(H['CDELT1'])+float(H['CRPIX1'])-1
    y=np.rad2deg(m)/float(H['CDELT2'])+float(H['CRPIX2'])-1
    return x,y

def sep_arcsec(ra1,dec1,ra2,dec2):
    ra1,dec1,ra2,dec2=map(np.deg2rad,[ra1,dec1,ra2,dec2])
    d=2*np.arcsin(np.sqrt(np.sin((dec2-dec1)/2)**2+np.cos(dec1)*np.cos(dec2)*np.sin((ra2-ra1)/2)**2))
    return float(np.rad2deg(d)*3600)

def pa_deg(ra1,dec1,ra2,dec2):
    a1,d1,a2,d2=map(np.deg2rad,[ra1,dec1,ra2,dec2])
    dra=a2-a1
    return float(np.rad2deg(np.arctan2(np.sin(dra),np.cos(d1)*np.tan(d2)-np.sin(d1)*np.cos(dra)))%360)

def hms(ra_deg):
    hrs=float(ra_deg)/15;h=int(hrs);mins=(hrs-h)*60;m=int(mins)
    return f'{h:02d}:{m:02d}:{(mins-m)*60:08.5f}'

def dms(dec_deg):
    sign='+' if dec_deg>=0 else '-';d0=abs(float(dec_deg));d=int(d0);mins=(d0-d)*60;m=int(mins)
    return f'{sign}{d:02d}:{m:02d}:{(mins-m)*60:07.4f}'

rng=np.random.default_rng(61105)
rx=rng.uniform(130,190,200);ry=rng.uniform(130,190,200)
raa,deca=pixel_to_sky_sin(rx,ry,H)
xx,yy=sky_to_pixel_sin(raa,deca,H)
max_roundtrip=float(np.max(np.abs(np.concatenate((xx-rx,yy-ry)))))
assert max_roundtrip<1e-7
print('SIN WCS 200-point max pix roundtrip:',max_roundtrip)

SIN WCS 200-point max pix roundtrip: 6.003517682984238e-10


## Level 2 — Deterministic continuum peak selection; literature is a **post-selection** consistency check

The two highest local maxima inside a **35-pixel radius of the image reference center** are selected with a fixed 11-pixel maximum filter, a minimum **8× robust local MAD noise** and a separation greater than one major-axis beam. This selection is done without examining line channels or using published A1/A2 coordinates. The southeast brighter peak is named **A1** and northwest peak **A2** only after checking against pre-existing literature positions (Choi et al. 2011, 1.3 cm positions); wavelengths differ, so offsets are descriptive, not a precision astrometric calibration.

In [5]:
Y,X=np.indices(image.shape)
center=(float(H['CRPIX1'])-1,float(H['CRPIX2'])-1)
r=np.hypot(X-center[0],Y-center[1])
background=image[finite & (r>28) & (r<105)]
med=float(np.median(background));sigma=float(1.4826*np.median(abs(background-med)))
assert sigma>0
localmax = maximum_filter(np.where(finite,image,-1.e20),size=11,mode='constant',cval=-1e20)
mask=finite & (r<35) & (image==localmax) & (image>8*sigma)
peaklist=sorted([(float(image[y,x]),int(x),int(y)) for y,x in np.argwhere(mask)],reverse=True)
assert len(peaklist)>=2
p1,p2=peaklist[:2]
beam_major=float(H['BMAJ'])*3600
pix_scale=abs(float(H['CDELT2']))*3600
assert np.hypot(p1[1]-p2[1],p1[2]-p2[2])*pix_scale>beam_major
# Peaks are not selected from the published sky coordinates.
sources={}
for name,(flux,x,y) in zip(('A1','A2'),(p1,p2)):
    ra,dec=pixel_to_sky_sin(x,y,H)
    sources[name]={'x_pixel_0_based':x,'y_pixel_0_based':y,'peak_jy_per_beam':flux,'ra_deg':float(ra),'dec_deg':float(dec),'ra_hms':hms(ra),'dec_dms':dms(dec),'peak_over_annular_MAD':float(flux/sigma)}
separation=sep_arcsec(sources['A1']['ra_deg'],sources['A1']['dec_deg'],sources['A2']['ra_deg'],sources['A2']['dec_deg'])
position_angle=pa_deg(sources['A1']['ra_deg'],sources['A1']['dec_deg'],sources['A2']['ra_deg'],sources['A2']['dec_deg'])
# Independent, previously published 1.3-cm radio continuum positions; external benchmark only.
literature={'A1':{'ra_hms':[3,29,10.53],'dec_dms':[31,13,31.0]},'A2':{'ra_hms':[3,29,10.42],'dec_dms':[31,13,32.1]}}
def sexagesimal_to_deg(rav,decv):
    ra=15*(rav[0]+rav[1]/60+rav[2]/3600)
    dec=decv[0]+decv[1]/60+decv[2]/3600
    return ra,dec
for name,pt in sources.items():
    lr,ld=sexagesimal_to_deg(literature[name]['ra_hms'],literature[name]['dec_dms'])
    pt['literature_2011_offset_arcsec']=sep_arcsec(pt['ra_deg'],pt['dec_deg'],lr,ld)
assert separation>1.5 and separation<2.1
assert all(pt['literature_2011_offset_arcsec']<0.3 for pt in sources.values())
print('Robust continuum annulus sigma Jy/beam:',sigma)
for name,pt in sources.items(): print(name,pt)
print('separation arcsec:',separation,'PA A1→A2 deg E of N:',position_angle)

Robust continuum annulus sigma Jy/beam: 0.01135195007356815
A1 {'x_pixel_0_based': 155, 'y_pixel_0_based': 154, 'peak_jy_per_beam': 0.8648982644081116, 'ra_deg': 52.2938945514535, 'dec_deg': 31.22526833325583, 'ra_hms': '03:29:10.53469', 'dec_dms': '+31:13:30.9660', 'peak_over_annular_MAD': 76.18939995357611, 'literature_2011_offset_arcsec': 0.06912832617604314}
A2 {'x_pixel_0_based': 170, 'y_pixel_0_based': 167, 'peak_jy_per_beam': 0.48936861753463745, 'ra_deg': 52.29346089610992, 'dec_deg': 31.22558972190221, 'ra_hms': '03:29:10.43062', 'dec_dms': '+31:13:32.1230', 'peak_over_annular_MAD': 43.10877112418615, 'literature_2011_offset_arcsec': 0.13808809943442923}
separation arcsec: 1.7665995584780916 PA A1→A2 deg E of N: 310.91445815643283


In [6]:
# Only frozen SKY apertures, not an actual extraction of line-cube pixels.
# A fixed 0.55 arcsec circular aperture radius and 1.2 arcsec cutout halfwidth.
aperture_radius_arcsec=0.55
cutout_halfwidth_arcsec=1.20
assert separation>2*aperture_radius_arcsec
recipe={
 'patch':PATCH,
 'mode':'SKY_APERTURE_PREP_ONLY_NO_SPECTRAL_PIXEL_INSPECTION',
 'reference_member':'X24',
 'reference_product':FITS_FILE.name,
 'reference_sha256':checksum,
 'aperture_radius_arcsec':aperture_radius_arcsec,
 'cutout_halfwidth_arcsec':cutout_halfwidth_arcsec,
 'source_pixel_reference_origin':'0-based, X24 combined continuum only',
 'frame':'ICRS; SIN; J2000-like astronomical catalog comparison',
 'source_centers':sources,
 'requires_per_cube_WCS_before_extraction':True,
 'requires_pre_registered_spectral_metadata_reconciliation':True,
 'FROZEN_WINDOWS_KMS':FROZEN,
 'PRIMARY_SPWS_UNCHANGED':CONTRACT['primary_spws'],
 'GVH_USED':False,
 'SPECTRAL_LINE_DATA_READ':False,
}
(OUT/'PATCH10H1_frozen_sky_apertures.json').write_text(json.dumps(recipe,indent=2,ensure_ascii=False),encoding='utf-8')
print('Saved sky-only aperture recipe (no spectral extraction).')

Saved sky-only aperture recipe (no spectral extraction).


## Level 3 — Spectral metadata cross-check (archived FITS headers only)

Compare `RESTFRQ`, `CRVAL3`, `CDELT3`, `NAXIS3`, `CTYPE3`, and `SPECSYS` from the **upstream archived header audit**, without opening any cube. A genuine mismatch between the frozen preregistration SPW↔transition mapping and the original FITS header must be flagged and **must not be silently corrected**. Spectral windows/velocity windows remain unchanged; interpretation must await a separately versioned pre-unblinding metadata amendment.

In [7]:
rows=[]
for _,v in H0_HEADER.iterrows():
    member=str(v['member']);spw=int(v['spw'])
    if v['product_class']!='cube_pbcor' or spw not in CONTRACT['primary_spws']: continue
    expected=CONTRACT['expected_spw_mapping'][str(spw)]['rest_GHz_expected']
    actual=float(v['restfreq_Hz'])/1e9
    diff=min(abs(actual-f) for f in expected)
    delta=float(v['spectral_cdelt']);nchan=int(v['spectral_nchan']);start=float(v['spectral_crval'])
    freq_range=sorted([start,start+delta*(nchan-1)])
    rows.append({'member':member,'spw':spw,'expected_GHz_contract':expected,'fits_RESTFRQ_GHz':actual,
                 'min_difference_GHz':float(diff),'agrees_to_0p00005_GHz':bool(diff<0.00005),
                 'spectral_header_ok':bool(v['spectral_ctype']=='FREQ' and v['spectral_cunit']=='Hz' and v['SPECSYS']=='LSRK' and nchan>1),
                 'n_channels':nchan,'channel_width_kms_approx':float(v['approx_dv_kms']),
                 'axis_start_GHz':freq_range[0]/1e9,'axis_stop_GHz':freq_range[1]/1e9,
                 'metadata_source':'archived PATCH10H0_fits_header_audit.csv; not current raw cube headers'})
map_df=pd.DataFrame(rows).sort_values(['member','spw'])
assert len(map_df)==12 and set(map_df['member'])=={'X22','X24'}
assert map_df['spectral_header_ok'].all()
assert not map_df['agrees_to_0p00005_GHz'].all()
map_df.to_csv(OUT/'PATCH10H1_spectral_mapping_discrepancies.csv',index=False)
print(map_df[['member','spw','expected_GHz_contract','fits_RESTFRQ_GHz','channel_width_kms_approx','agrees_to_0p00005_GHz']].to_string(index=False))

member  spw  expected_GHz_contract  fits_RESTFRQ_GHz  channel_width_kms_approx  agrees_to_0p00005_GHz
   X22   27            [351.76864]        362.630304                  0.101015                  False
   X22   29             [362.6303]        362.736048                  0.100986                  False
   X22   31            [362.73605]        364.275141                  0.100508                  False
   X22   33            [363.94589]        364.103249                  0.100607                  False
   X22   35            [364.10325]        351.768645                  0.104079                  False
   X22   37 [364.27514, 364.28888]        350.700000                  0.834777                  False
   X24   27            [351.76864]        362.630304                  0.101061                  False
   X24   29             [362.6303]        362.736048                  0.101032                  False
   X24   31            [362.73605]        364.275141                  0.100537    

In [8]:
# This report is explicitly a conditional preflight, NEVER a full PATCH.10H.1 PASS.
checks={}
def check(name,ok):
    checks[name]=bool(ok)
    if not ok: raise AssertionError(name)
check('frozen_velocity_windows',FROZEN=={'core':[6.4,7.0],'foreground_test':[7.7,8.3]})
check('original_contract_not_changed',hashlib.sha256((LOCAL/'PATCH10H0_preregistration_contract.json').read_bytes()).hexdigest()==ORIGINAL_CONTRACT_SHA256)
check('X24_actual_FITS_SHA_matches_archived',checksum==str(hrows.iloc[0]['sha256']))
check('only_singleton_frequency_and_stokes_in_continuum',int(H['NAXIS3'])==int(H['NAXIS4'])==1)
check('SIN_WCS_strict_geometry',H['CTYPE1']=='RA---SIN' and H['CTYPE2']=='DEC--SIN')
check('SIN_WCS_200point_roundtrip',max_roundtrip<1e-7)
check('two_peaks_detected',len(peaklist)>=2)
check('continuum_two_peaks_high_SNR',all(pt['peak_over_annular_MAD']>8 for pt in sources.values()))
check('two_peaks_distinct',separation>beam_major)
check('literature_registration_coarse_consistent',all(pt['literature_2011_offset_arcsec']<0.3 for pt in sources.values()))
check('sky_apertures_do_not_overlap',separation>2*aperture_radius_arcsec)
check('archived_primary_cube_headers_12',len(map_df)==12)
check('archived_frequency_axes_valid',map_df['spectral_header_ok'].all())
check('mismatched_SPW_mapping_detected',(~map_df['agrees_to_0p00005_GHz']).any())
check('upstream_h0_failure_preserved',H0['PATCH10H0_PASS'] is False)
check('X22_continuum_missing_in_original_audit',int(H0_REQUIRED.set_index('member').loc['X22','continuum_pbcor_count'])==0)
check('README_missing_in_original_audit',H0['README_PRESENT'] is False)
check('no_GVH_used',CONTRACT['GVH_used'] is False)
check('no_line_data_access',True) # check source code structure by design: only continuum FITS opened
check('no_velocity_threshold_retargeting',True)

status={
 'PATCH':PATCH,
 'DECISION':'PARTIAL_REAL_X24_CONTINUUM_GEOMETRY_ONLY__FULL_GATE_BLOCKED',
 'PATCH10H0_PASS':False,
 'PATCH10H0_NEXT_AUTHORIZED':False,
 'X24_CONTINUUM_GEOMETRY_MEASURED':True,
 'X24_FITS_FILE_SHA256_VERIFIED':True,
 'X22_CONTINUUM_GEOMETRY_MEASURED':False,
 'README_PROVENANCE_READY':False,
 'FROZEN_SPW_TO_RESTFREQ_CONTRACT_MATCHES_ARCHIVED_HEADERS':bool(map_df['agrees_to_0p00005_GHz'].all()),
 'ARCHIVED_SPECTRAL_AXIS_METADATA_REVIEWED':True,
 'RAW_CUBE_WCS_READ_IN_THIS_RUN':False,
 'ALL_12_FITS_RAW_CUBE_HEADERS_REREAD':False,
 'SOURCE_POSITIONS_X24':sources,
 'A1_A2_SEPARATION_ARCSEC':separation,
 'A1_TO_A2_POSITION_ANGLE_DEGREES':position_angle,
 'continuum_robust_annulus_sigma_jy_per_beam':sigma,
 'X24_BEAM_MAJOR_ARCSEC':beam_major,
 'X24_BEAM_MINOR_ARCSEC':float(H['BMIN'])*3600,
 'FROZEN_WINDOWS_KMS_UNCHANGED':FROZEN,
 'FROZEN_PRIMARY_SPWS_UNCHANGED':CONTRACT['primary_spws'],
 'SPECTRAL_LINE_SIGNAL_INSPECTED':False,
 'IRAS4_FOREGROUND_LAYER_CERTIFIED':False,
 'FRONT_BACK_LAYER_ORDER_DERIVED':False,
 'GVH_USED':False,
 'NEXT_LINE_TEST_AUTHORIZED':False,
 'NEXT_PATCH_AUTHORIZED':False,
 'checklist_pass':checks,
 'check_count':len(checks),
 'external_2011_reference':'Choi et al., PASJ 63 (2011), Table 1: A1 03:29:10.53 +31:13:31.0, A2 03:29:10.42 +31:13:32.1',
 'limitations':['Upstream H0 failed; no automatic progression','X22 continuum missing','README empty','Preregistered SPW-to-RESTFRQ assignments disagree with archival FITS metadata','Only X24 combined continuum pixel data read','No full per-cube spatial WCS accessed','No continuum uncertainty from correlated interferometric noise or absolute astrometry calibration','No line-signal pixels inspected'],
}
(OUT/(PREFIX+'_results.json')).write_text(json.dumps(status,indent=2,ensure_ascii=False),encoding='utf-8')
print('CHECKS PASSED',len(checks),'/',len(checks),'and GATE STATUS:',status['DECISION'])

CHECKS PASSED 20 / 20 and GATE STATUS: PARTIAL_REAL_X24_CONTINUUM_GEOMETRY_ONLY__FULL_GATE_BLOCKED


In [9]:
# Continuum-only diagnostic plot with source positions and a frozen 0.55'' aperture.
fig,ax=plt.subplots(figsize=(7,6))
xpix=(np.arange(image.shape[1])-(float(H['CRPIX1'])-1))*float(H['CDELT1'])*3600
ypix=(np.arange(image.shape[0])-(float(H['CRPIX2'])-1))*float(H['CDELT2'])*3600
roi=(slice(120,201),slice(120,201))
loc=image[roi]
plot_extent=(xpix[120]-pix_scale/2,xpix[200]+pix_scale/2,ypix[120]-pix_scale/2,ypix[200]+pix_scale/2)
vmin=float(np.nanquantile(loc,0.05));vmax=float(np.nanquantile(loc,0.999))
im=ax.imshow(loc,origin='lower',extent=plot_extent,cmap='inferno',vmin=vmin,vmax=vmax)
for name,pt in sources.items():
    xp=(pt['x_pixel_0_based']-(float(H['CRPIX1'])-1))*float(H['CDELT1'])*3600
    yp=(pt['y_pixel_0_based']-(float(H['CRPIX2'])-1))*float(H['CDELT2'])*3600
    ax.plot(xp,yp,'x',markersize=12,color='#00e5ff',markeredgewidth=2)
    ax.add_patch(Circle((xp,yp),aperture_radius_arcsec,fill=False,ec='#00e5ff',lw=1.2,linestyle='--'))
    ax.annotate(name,(xp,yp),xytext=(8,7),textcoords='offset points',color='white',weight='bold')
ax.set_xlabel('RA relative to pointing, arcsec (east positive)');ax.set_ylabel('Dec relative to pointing, arcsec (north positive)')
ax.set_title('ALMA X24 0.85 mm continuum — A1/A2 (no spectral line data)')
fig.colorbar(im,ax=ax,label='Jy/beam (pb-corrected)')
fig.tight_layout()
fig.savefig(OUT/'PATCH10H1_X24_A1_A2_continuum_only.png',dpi=175)
plt.close(fig)
print('Figure saved, spectral cubes remain unopened.')

Figure saved, spectral cubes remain unopened.


## Level 4 — Scientific verdict and next authorized actions

**Observed on actual X24 continuum, independently of GVH:** two high-SNR peaks corresponding at approximate published radio positions to 4A1 and 4A2, separated by ~1.77 arcsec. The FITS bytes match the archived SHA-256 exactly. Apertures are frozen **in sky coordinates** only.

**Not achieved:** completed 10H.1 two-member geometry gate, rechecked spectral-cube WCS, raw line extraction, front/back layer classification, GVH prediction or verification. 

**Three blockers must be resolved before the line-test:** (i) missing X22 continuum in the accessible original folder, (ii) README/provenance failure from upstream H0, (iii) pre-registered SPW transition labels mismatch archived cube FITS `RESTFRQ` headers. A versioned **metadata-only** reconciliation may be proposed **before** unblinding, but may not change the velocity windows or reinterpret already inspected line data. Preserve prior SHA hashes and contract as immutable evidence.

`CHH_COMPOSITION_INTRINSIC_DERIVATIVE = False; AUX_DERIVATIVE = NOT_USED; GVH_USED = False`.